# Short-Trigger Revival Diagnostic (Step 1) — does a long event hide shorter episodes?

**Status:** diagnostic · validated on representative counties (not the full catalog) · **no price change**
**Reads with:** [`06_short_trigger_frequency_recovery.md`](../../docs/dicsscssion/eventization_frequency_contract/06_short_trigger_frequency_recovery.md) · [`04_duration_conservatism.md`](../../docs/dicsscssion/eventization_frequency_contract/04_duration_conservatism.md) · [`cell_read_fundamentals.md`](../../docs/methodology/02_per_customer/cell_read_fundamentals.md)

The per-customer rate aggregates county outages. A long county event can be **(a)** one coherent
outage, or **(b)** several shorter outages merged by aggregation. The same mechanism cuts both ways:

```
 ONE mechanism — aggregation pushes exposure UP the duration axis:
   short clustered outages ──merge──► one long county event
     8h+ : the long event exists largely because of the merge  → OVER-fed  (the cushion)
     2-4h: the shorts get absorbed → fewer short events counted → UNDER-fed (the verify zone)
```

So the question behind **"≥8h is established"** is concrete and measurable: *do ≥8h events hide a
**second** ≥8h surge?* If they don't, the 8h frequency isn't under-counted by hidden episodes. This
notebook measures it from the raw 15-min paths.

In [1]:
import os, numpy as np, pandas as pd
while not os.path.isdir("price_engine/data/raw") and os.getcwd() != "/":
    os.chdir("..")
print("cwd:", os.getcwd())

RAW = "price_engine/data/raw"
GAP_NS = np.int64(45 * 60 * 1_000_000_000)   # 45-min gap tolerance (matches eagle-i-45min)
NS_PER_H = 3.6e12
TS = [2, 4, 8, 12, 24]
BLOCKS = "▁▂▃▄▅▆▇█"
# representative counties: dense storm-driven (Erie/Worcester/Suffolk), humid-subtropical (Alachua),
# sparse/flat (Harney), source-gap (Concho). Few FIPS -> cheap and inspectable.
FIPS = {36029: "Erie NY", 25027: "Worcester MA", 25025: "Suffolk MA",
        12001: "Alachua FL", 41025: "Harney OR", 48095: "Concho TX"}
YEARS = [2021, 2022, 2023]   # a representative window; the method scales to the full catalog

cwd: /Users/divy/code/work/infrasure_git_codes/outage_pricing


## Method

Rebuild each county's events from the raw 15-min snapshots (45-min gap walk — matches `eagle-i-45min`).
Inside an event's customers-out curve, a **surge** = a run above a cutoff level; a **distinct ≥T
episode** = a surge lasting ≥ T. Pricing counts a qualifying event **once**; if an event holds two
separate ≥8h surges, that is a hidden *extra* 8h episode. The cutoff level is the open knob — we test
the event **mean** and **10 / 20 / 33 % of the event peak**.

In [2]:
def load_filtered(years, fips_set):
    parts = []
    for y in years:
        path = f"{RAW}/eaglei_outages_{y}.csv"
        if not os.path.exists(path):
            continue
        df = None
        for cols in (["fips_code", "customers_out", "run_start_time"],
                     ["fips_code", "sum", "run_start_time"]):
            try:
                df = pd.read_csv(path, usecols=cols, low_memory=False); break
            except (ValueError, KeyError):
                df = None
        df = df.rename(columns={"sum": "customers_out"})
        df["fips"] = pd.to_numeric(df["fips_code"], errors="coerce")
        df = df[df["fips"].isin(fips_set)]
        if df.empty:
            continue
        df["c"] = pd.to_numeric(df["customers_out"], errors="coerce")
        dt = pd.to_datetime(df["run_start_time"], errors="coerce", utc=True)
        df["t"] = dt.dt.tz_convert(None).values.astype("datetime64[ns]").view("int64")  # force ns
        df = df.dropna(subset=["fips", "t", "c"]); df = df[df["c"] > 0]
        parts.append(df[["fips", "t", "c"]])
    d = pd.concat(parts, ignore_index=True)
    d["fips"] = d["fips"].astype(int)
    return d.sort_values(["fips", "t"]).drop_duplicates(["fips", "t"], keep="last").reset_index(drop=True)


def split_events(t, c):
    # yield (duration_h, t_arr, c_arr) per event for one sorted FIPS stream
    if len(t) == 0:
        return
    newev = np.empty(len(t), bool); newev[0] = True
    newev[1:] = np.diff(t) > GAP_NS
    idx = list(np.where(newev)[0]) + [len(t)]
    for a, b in zip(idx[:-1], idx[1:]):
        tt, cc = t[a:b], c[a:b]
        yield (tt[-1] - tt[0]) / NS_PER_H + 0.25, tt, cc


def surge_durations(tt, cc, L):
    # durations (h) of maximal runs where customers-out >= L, from real timestamps
    m = cc >= L
    if not m.any():
        return []
    d = np.diff(np.concatenate(([0], m.astype(np.int8), [0])))
    return [(tt[e - 1] - tt[s]) / NS_PER_H + 0.25
            for s, e in zip(np.where(d == 1)[0], np.where(d == -1)[0])]


def spark(cc, width=56):
    if len(cc) > width:
        cc = np.array([b.max() for b in np.array_split(cc, width)])
    lo, hi = cc.min(), cc.max()
    if hi <= lo:
        return BLOCKS[0] * len(cc)
    return "".join(BLOCKS[i] for i in ((cc - lo) / (hi - lo) * 7).round().astype(int))


d = load_filtered(YEARS, set(FIPS))
evs = {f: list(split_events(d[d.fips == f]["t"].values.astype("int64"),
                            d[d.fips == f]["c"].values.astype(float))) for f in FIPS}
print(f"{len(d):,} snapshots · {d.fips.nunique()} counties · years {YEARS[0]}-{YEARS[-1]}")

277,644 snapshots · 6 counties · years 2021-2023


## Finding 1 — hidden ≥8h episodes are ~0%, and it's robust to the cutoff

`multi%` = share of ≥8h events that contain a **second** ≥8h surge (a genuinely hidden extra episode).
Across **every** cutoff it is ~0% — a long event almost never hides a second long outage, so the 8h
frequency is not under-counted by hidden episodes. (`frag%` = qualifying events with no single sustained
≥8h run; it is high because the 15-min curve is choppy — the wrinkle discussed at the end. It is a
property of the level-cutoff, not a threat to the multi% result.)

In [3]:
CUTOFFS = {"mean": lambda cc: cc.mean(), "10% peak": lambda cc: 0.10 * cc.max(),
           "20% peak": lambda cc: 0.20 * cc.max(), "33% peak": lambda cc: 0.33 * cc.max()}
STORM = [36029, 25027, 25025, 12001]   # the well-covered, high-volume counties
T = 8
print(f"{'cutoff':>9} {'8h events':>10} {'multi% (≥2 surges ≥8h)':>24} {'frag%':>7}")
for cut, fn in CUTOFFS.items():
    base = multi = frag = 0
    for f in STORM:
        for dur, tt, cc in evs[f]:
            if dur >= T:
                base += 1
                s = sum(1 for x in surge_durations(tt, cc, fn(cc)) if x >= T)
                multi += (s >= 2); frag += (s == 0)
    print(f"{cut:>9} {base:>10,} {multi/base*100:>23.0f}% {frag/base*100:>6.0f}%")

   cutoff  8h events   multi% (≥2 surges ≥8h)   frag%
     mean      2,469                       0%     93%


 10% peak      2,469                       0%     76%
 20% peak      2,469                       0%     85%


 33% peak      2,469                       0%     91%


## Finding 2 — the cutoff cleanly separates one coherent surge from two

Erie's longest event is **one** coherent surge plus a long restoration tail; Suffolk's is **two**
genuine humps. ~10–20 % of the event peak reads Erie as 1 and Suffolk as 2; 33 % is too high (it loses
Suffolk's second hump). So the method **isn't blind** — it finds two when two exist, which means the
~0 % above is real rarity, not a failure to detect.

In [4]:
erie = max(evs[36029], key=lambda e: e[0])
suff = max(evs[25025], key=lambda e: sum(1 for x in surge_durations(e[1], e[2], 0.2 * e[2].max()) if x >= 8))
for label, (dur, tt, cc) in [("Erie — coherent", erie), ("Suffolk — two-hump", suff)]:
    print(f"\n{label}:  dur={dur:.0f}h  peak={int(cc.max()):,}  mean={int(cc.mean()):,}")
    print("  " + spark(cc))
    for cut, fn in CUTOFFS.items():
        n8 = sum(1 for x in surge_durations(tt, cc, fn(cc)) if x >= 8)
        print(f"    {cut:>9}: {n8} surge(s) ≥8h")


Erie — coherent:  dur=139h  peak=34,999  mean=15,242
  ▅▇▇██▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▆▆▆▄▄▃▃▃▄▄▄▅▄▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
         mean: 1 surge(s) ≥8h
     10% peak: 1 surge(s) ≥8h
     20% peak: 1 surge(s) ≥8h
     33% peak: 1 surge(s) ≥8h

Suffolk — two-hump:  dur=86h  peak=128  mean=26
  ▁▁▁▁▁▁▁▁▁██████▁▁▁▁▁▁▁▁▁▁▄▄▄▄▃▃▃▃▃▃▃▁▁▁▁▁▁▃▅▅▄▃▃▃▃▃▃▃▃▃▃
         mean: 2 surge(s) ≥8h
     10% peak: 2 surge(s) ≥8h
     20% peak: 2 surge(s) ≥8h
     33% peak: 0 surge(s) ≥8h


## What this establishes — and what it defers

- **≥8h is established.** Genuine hidden 8h episodes are ~0 %, robust to the cutoff (on these
  representative counties). Combined with the A011 conservative cushion (the mean over-states
  per-customer exposure ~2–3× at long durations), **≥8h is well-cushioned**. This is the evidence
  behind the dashboard's posture gate — nothing on the dashboard changes.
- **Deferred — the short-trigger *recovery* build.** A pure level-cutoff **fragments** choppy curves
  (the high `frag%`), so *counting / re-pricing* short-trigger episodes needs a second knob: a
  **dip tolerance** — how long the curve must stay below the line to count as a real break (the surge
  analogue of the 45-min event gap tolerance) — plus scaling beyond these counties. That is the
  deferred quant build (gated on business need for short triggers); it does **not** move the price today.

**Scope honesty:** 6 representative counties × 3 years — enough to validate the method and the ≥8h
conclusion's robustness, not a national rate. The full-catalog confirmation is the scale-up.